# 16S rPCA Analyses of the Longitudinal Acne Study

Date last updated: 1/7/25

Notebook author: Britta De Pessemier, Yang Chen

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:
- rPCA beta diversity analysis for 16S data

In [1]:
# import Python packages
import pandas as pd
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
import scipy
import scipy.stats as ss
from skbio.stats.distance import permanova
from skbio.stats.ordination import OrdinationResults
import biom
from biom import load_table
from gemelli.rpca import rpca
from matplotlib.patches import Ellipse
from skbio.stats.distance import permanova, DistanceMatrix
from itertools import combinations
import qiime2 as q2
from qiime2 import Artifact
from qiime2 import Metadata
from qiime2.plugins.feature_table.methods import filter_samples
from statsmodels.stats.multitest import multipletests
import os

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)

# Set display precision globally
pd.set_option('display.float_format', '{:.10f}'.format)

## Functions from Gemelli

In [2]:
# ----------------------------------------------------------------------------
# Copyright (c) 2016-2023, QIIME 2 development team.
#
# Distributed under the terms of the Modified BSD License.
#
# The full license is in the file LICENSE, distributed with this software.
# ----------------------------------------------------------------------------

# Note: RPCA functions are from https://github.com/qiime2/q2-feature-table/blob/dev/q2_feature_table/_filter.py; an alternative to direct import from my qiime2 environment
def _validate_nonempty_table(table):
    if table.is_empty():
        raise ValueError("The resulting table is empty. This can happen if "
                         "you filter all samples or features out of the "
                         "table. Please check your filtering parameters and "
                         "try again.")


def _get_biom_filter_function(ids_to_keep, min_frequency, max_frequency,
                              min_nonzero, max_nonzero):
    ids_to_keep = set(ids_to_keep)
    if max_frequency is None:
        max_frequency = np.inf
    if max_nonzero is None:
        max_nonzero = np.inf

    def f(data_vector, id_, metadata):
        return (id_ in ids_to_keep) and \
               (min_frequency <= data_vector.sum() <= max_frequency) and \
               (min_nonzero <= (data_vector > 0).sum() <= max_nonzero)
    return f


_other_axis_map = {'sample': 'observation', 'observation': 'sample'}


def _filter_table(table, min_frequency, max_frequency, min_nonzero,
                  max_nonzero, metadata, where, axis, exclude_ids=False,
                  filter_opposite_axis=True,
                  allow_empty_table=False):
    if min_frequency == 0 and max_frequency is None and min_nonzero == 0 and\
       max_nonzero is None and metadata is None and where is None and\
       exclude_ids is False:
        raise ValueError("No filtering was requested.")
    if metadata is None and where is not None:
        raise ValueError("Metadata must be provided if 'where' is "
                         "specified.")
    if metadata is None and exclude_ids is True:
        raise ValueError("Metadata must be provided if 'exclude_ids' "
                         "is True.")
    if metadata is not None:
        ids_to_keep = metadata.get_ids(where=where)
    else:
        ids_to_keep = table.ids(axis=axis)
    if exclude_ids is True:
        ids_to_keep = set(table.ids(axis=axis)) - set(ids_to_keep)

    filter_fn1 = _get_biom_filter_function(
        ids_to_keep, min_frequency, max_frequency, min_nonzero, max_nonzero)
    table.filter(filter_fn1, axis=axis, inplace=True)

    # filter on the opposite axis to remove any entities that now have a
    # frequency of zero
    if filter_opposite_axis:
        table.remove_empty(axis=_other_axis_map[axis], inplace=True)

    if not allow_empty_table:
        _validate_nonempty_table(table)


def filter_samples(table: biom.Table, min_frequency: int = 0,
                   max_frequency: int = None, min_features: int = 0,
                   max_features: int = None,
                   metadata: q2.Metadata = None, where: str = None,
                   exclude_ids: bool = False,
                   filter_empty_features: bool = True,
                   allow_empty_table: bool = False)\
                  -> biom.Table:
    _filter_table(table=table, min_frequency=min_frequency,
                  max_frequency=max_frequency, min_nonzero=min_features,
                  max_nonzero=max_features, metadata=metadata,
                  where=where, axis='sample', exclude_ids=exclude_ids,
                  filter_opposite_axis=filter_empty_features,
                  allow_empty_table=allow_empty_table
                  )
    return table

## Metadata organization

In [3]:
# Read in metadata and preprocess
md_path = '../Metadata/metadata_final_22102024.tsv'
md = pd.read_csv(md_path, sep='\t')

# Subset to skin samples
md = md.loc[md['sample_type'] == 'skin'].copy()

# Helper for numeric coercion
def clean_numeric(cols):
    return (
        md[cols]
        .replace('not applicable', np.nan)
        .apply(pd.to_numeric, errors='coerce')
        .fillna(0)
        .sum(axis=1)
    )

# Severity-related metrics
cheek_cols = [
    'visual_assessment_on_photography_acne_severity_cheek_left',
    'visual_assessment_on_photography_acne_severity_cheek_right'
]

md['local_lesion_severity'] = clean_numeric(cheek_cols)

# Same definition kept explicit for clarity
md['acne_severity'] = md['local_lesion_severity']

md['inflammatory_lesions_face'] = clean_numeric(
    ['visual_assessment_in_vivo_number_of_inflammatory_lesions_face']
)

md['noninflammatory_lesions_face'] = clean_numeric(
    ['visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face']
)

md['a'] = clean_numeric(
    ['skincam_a_cheek_left', 'skincam_a_cheek_right']
)

# Cohort, subject IDs, and class
srn = pd.to_numeric(md['subject_randomization_number'], errors='coerce').astype('Int64')

md['cohort'] = np.where(srn > 299, 'acne', 'control')
md['subject_randomization_id'] = 'RD' + srn.astype(str).str.zfill(3)
md['class'] = np.where(md['acne_severity'] >= 1, 'acne', 'healthy')

# Category assignment logic
md['category'] = np.select(
    [
        md['c_zone'].eq('C3'),
        md['c_zone'].isin(['C1', 'C2']) & md['cohort'].eq('acne'),
        md['c_zone'].isin(['C1', 'C2']) & md['cohort'].eq('control')
    ],
    ['clear_zone', 'acne', 'control'],
    default=np.nan
)

# Human-readable category labels
md['category_label'] = md['category'].map({
    'clear_zone': 'Acne Non-lesional',
    'acne': 'Acne Lesional',
    'control': 'Healthy'
})

# Lesional severity bins (vectorized)
sev = md['local_lesion_severity']

md['lesional_score'] = np.select(
    [
        md['category_label'].eq('Acne Lesional') & sev.isin([1, 2]),
        md['category_label'].eq('Acne Lesional') & sev.isin([3, 4]),
        md['category_label'].eq('Acne Lesional') & sev.isin([5, 6]),
    ],
    ['Low', 'Moderate', 'High'],
    default=np.nan
)

# Ensure boolean columns are strings for metadata compatibility
bool_cols = md.select_dtypes(include='bool').columns
md[bool_cols] = md[bool_cols].astype(str)

md = md.set_index('SampleID')
md.index.name = '#SampleID'

In [4]:
# Make qiime2 compatible md
md_q2 = Metadata(md)

## Read in feature tables

In [5]:
# Load in tables
v3_table = Artifact.load(
    '../Data/16S/Tables/from_Qiita/179426_feature-table_16S_V1V3_rare-11054.qza'
)
v4_table = Artifact.load(
    '../Data/16S/Tables/from_Qiita/174951_feature-table_16S_V4_rare-3769.qza'
)

# Convert qiime2 table artifacts to BIOM
v3_table = v3_table.view(biom.Table)
v4_table = v4_table.view(biom.Table)

# Normalize BIOM sample IDs to metadata format
def normalize_sample_id(sample_id, drop_prefix=None):
    # Drop BLANK samples
    if 'BLANK' in sample_id.upper():
        return None

    # Replace underscores with dots first (so prefixes match)
    sample_id = sample_id.replace('_', '.')

    # Drop run-specific prefix if provided
    if drop_prefix and sample_id.startswith(drop_prefix):
        sample_id = sample_id[len(drop_prefix):]

    return sample_id

def rename_biom_sample_ids(table, id_func):
    # First drop samples that should be removed (e.g., BLANKs)
    keep_ids = []
    id_map = {}

    for sid in table.ids(axis='sample'):
        new_id = id_func(sid)

        if new_id is None:
            continue  # drop this sample
        else:
            keep_ids.append(sid)
            id_map[sid] = new_id

    # Filter table to kept samples only
    table = table.filter(
        keep_ids,
        axis='sample',
        inplace=False
    )

    # Now safely update IDs (no missing mappings)
    table = table.update_ids(
        id_map,
        axis='sample',
        inplace=False
    )

    return table



# V1–V3: drop 173980.14901.
v3_table = rename_biom_sample_ids(
    v3_table,
    lambda s: normalize_sample_id(s, drop_prefix="173980.14901.")
)

# V4: drop 173620.14901.
v4_table = rename_biom_sample_ids(
    v4_table,
    lambda s: normalize_sample_id(s, drop_prefix="173620.14901.")
)


In [6]:
# Sanity checks
print("Metadata IDs:", list(md_q2.to_dataframe().index)[:5])
print("V3 BIOM IDs:", v3_table.ids()[:5])
print("V4 BIOM IDs:", v4_table.ids()[:5])

assert len(md_q2.to_dataframe()) > 0
assert len(set(v3_table.ids()) & set(md_q2.to_dataframe().index)) > 0
assert len(set(v4_table.ids()) & set(md_q2.to_dataframe().index)) > 0


Metadata IDs: ['LAMI.RD308.D16.C1', 'LAMI.RD310.D21.C1', 'LAMI.RD305.D21.C3', 'LAMI.RD306.D18.C2', 'LAMI.RD306.D7.C2']
V3 BIOM IDs: ['LAMI.RD001.D0.C1' 'LAMI.RD001.D0.C2' 'LAMI.RD001.D14.C1'
 'LAMI.RD001.D28.C1' 'LAMI.RD002.D14.C1']
V4 BIOM IDs: ['LAMI.RD001.D0.C1' 'LAMI.RD001.D14.C1' 'LAMI.RD004.D0.C2'
 'LAMI.RD001.D0.C2' 'LAMI.RD004.D28.C1']


In [7]:
def intersect_and_filter_tables(
    v3_table,
    v4_table,
    md_q2,
    axis='sample'
):
    """
    Intersect sample IDs across V1–V3 and V4 BIOM tables,
    then filter and reorder both tables and metadata identically.
    """

    # Extract IDs
    shared_ids = (
        set(v3_table.ids(axis=axis))
        & set(v4_table.ids(axis=axis))
        & set(md_q2.to_dataframe().index)
    )

    if len(shared_ids) == 0:
        raise ValueError("No shared sample IDs found.")

    # Define canonical order (sorted is safest & reproducible)
    shared_ids = sorted(shared_ids)

    # Filter BIOM tables
    v3_filt = v3_table.filter(shared_ids, axis=axis, inplace=False)
    v4_filt = v4_table.filter(shared_ids, axis=axis, inplace=False)

    # Explicitly reorder BIOM tables
    v3_filt = v3_filt.sort_order(shared_ids, axis=axis)
    v4_filt = v4_filt.sort_order(shared_ids, axis=axis)

    # Filter + reorder metadata
    md_df = md_q2.to_dataframe().loc[shared_ids].copy()
    md_filt = Metadata(md_df)

    # Content sanity checks (order-agnostic)
    assert set(v3_filt.ids(axis=axis)) == set(shared_ids)
    assert set(v4_filt.ids(axis=axis)) == set(shared_ids)
    assert set(md_df.index) == set(shared_ids)

    print(f"Shared samples retained: {len(shared_ids)}")

    return v3_filt, v4_filt, md_filt


In [8]:
v3_table, v4_table, md_q2 = intersect_and_filter_tables(
    v3_table,
    v4_table,
    md_q2
)

Shared samples retained: 184


## Plotting preparation

In [9]:
# Convert QIIME2 metadata to pandas once
md_df = md_q2.to_dataframe()

# Align metadata to each table
meta_v3 = md_df.loc[v3_table.ids(axis='sample')].copy()
meta_v4 = md_df.loc[v4_table.ids(axis='sample')].copy()

# Add derived columns
meta_v3['c_zone_group'] = meta_v3['cohort'] + '_' + meta_v3['c_zone']
meta_v3['id_loc'] = meta_v3['subject_randomization_id'] + '_' + meta_v3['c_zone']

meta_v4['c_zone_group'] = meta_v4['cohort'] + '_' + meta_v4['c_zone']
meta_v4['id_loc'] = meta_v4['subject_randomization_id'] + '_' + meta_v4['c_zone']

# Convert to QIIME2 Metadata
# (bool → str is required by QIIME2)
q2_meta_v3 = Metadata(
    meta_v3.applymap(lambda x: str(x) if isinstance(x, bool) else x)
)

q2_meta_v4 = Metadata(
    meta_v4.applymap(lambda x: str(x) if isinstance(x, bool) else x)
)

# Define colors and shapes for severity groups
# Set the color palette for the groups in the correct order
group_colors = {
    'Healthy': '#3333B3',
    'Acne Non-lesional': '#5cbccb',
    'Acne Lesional': '#f16c52'
}

# Define colors for severity levels
severity_colors = {
    'Low': '#F1948A',
    'Moderate': '#EC7063',
    'High': '#C0392B'
}

group_order = ["Healthy", "Acne Non-lesional", "Acne Lesional"]

# Define tables, metadata, plot titles, file suffixes, and output paths
tables_metadata = [
    (v3_table, q2_meta_v3, 'RPCA - 16S rRNA (V1-V3)', 'V1-V3', '../Figures/Supplementary/Suppl_Figure_3/'),
    (v4_table, q2_meta_v4, 'RPCA - 16S rRNA (V4)', 'V4', '../Figures/Supplementary/Suppl_Figure_3/')
]

# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor

    ell = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=angle,
        edgecolor=color,
        facecolor=color,
        alpha=0.2,
        linewidth=0.5
    )
    ax.add_patch(ell)


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_85344/535492796.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  meta_v3.applymap(lambda x: str(x) if isinstance(x, bool) else x)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_85344/535492796.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  meta_v4.applymap(lambda x: str(x) if isinstance(x, bool) else x)


## Run RPCA

In [10]:
def run_rpca_analysis(table, metadata, group_column='category_label'):
    """
    Run RPCA, align metadata and distance matrix, and perform pairwise PERMANOVA
    with multiple-testing correction (Benjamini–Hochberg FDR).

    Returns
    -------
    ordination : OrdinationResults
    distance_matrix : DistanceMatrix
    spca_df : pd.DataFrame
    results_df : pd.DataFrame
        Includes raw p-values ('p-value') and FDR-corrected p-values ('p_adj')
    """

    np.seterr(divide='ignore')

    # Run RPCA
    ordination, distance = rpca(table)

    # Convert metadata to DataFrame
    metadata_df = metadata.to_dataframe()

    # Align metadata to distance matrix IDs (distance is authoritative)
    metadata_df = metadata_df.loc[
        metadata_df.index.intersection(distance.ids)
    ]

    if metadata_df.empty:
        raise ValueError("No overlapping sample IDs between metadata and distance matrix.")

    # Filter distance matrix
    metadata_df.index.name = None
    distance_matrix = distance.filter(metadata_df.index, strict=True)

    # Build ordination dataframe
    spca_df = pd.DataFrame(
        ordination.samples,
        index=ordination.samples.index,
        columns=['PC1', 'PC2', 'PC3']
    )

    spca_df[group_column] = spca_df.index.map(metadata_df[group_column])
    spca_df['lesional_score'] = spca_df.index.map(metadata_df['lesional_score'])

    # Pairwise PERMANOVA
    groups = metadata_df[group_column].dropna().unique()
    results = []

    for g1, g2 in combinations(groups, 2):
        subset_ids = metadata_df[
            metadata_df[group_column].isin([g1, g2])
        ].index

        if len(subset_ids) < 2:
            continue

        subset_distance = distance_matrix.filter(subset_ids, strict=True)
        subset_metadata = metadata_df.loc[subset_ids]

        perm = permanova(
            subset_distance,
            subset_metadata[group_column]
        )

        results.append({
            'Group1': g1,
            'Group2': g2,
            'Test Statistic': perm['test statistic'],
            'p-value': perm['p-value']
        })

    results_df = pd.DataFrame(results)

    # Multiple-testing correction (FDR)
    if not results_df.empty:
        results_df['p_adj'] = multipletests(
            results_df['p-value'],
            method='fdr_bh'
        )[1]

    return ordination, distance_matrix, spca_df, results_df

## Plot PCOA

In [11]:
def plot_rpca_results(
    ordination,
    spca_df,
    results_df,
    plot_title,
    output_path,
    file_suffix,
    group_order,
    group_colors,
    severity_colors
):
    """
    Plot RPCA results with ellipses, severity coloring, and PERMANOVA annotations.
    """

    os.makedirs(output_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=(7, 10))

    # Base scatter by category (exclude Acne Lesional; plotted via severity)
    for group in group_order:
        if group == "Acne Lesional":
            continue

        data = spca_df[spca_df['category_label'] == group]
        if data.empty:
            continue

        sns.scatterplot(
            data=data,
            x='PC1',
            y='PC2',
            facecolor=group_colors[group],
            edgecolor='k',
            linewidth=0.5,
            alpha=0.8,
            s=100,
            ax=ax,
            label=group
        )

    # Ellipses + stars
    for label, data in spca_df.groupby('category_label'):
        if len(data) < 3:
            continue

        points = data[['PC1', 'PC2']].values
        mean = points.mean(axis=0)
        cov = np.cov(points, rowvar=False)

        ellipse_color = {
            "Healthy": "#423fa6",
            "Acne Non-lesional": "#67b2be",
            "Acne Lesional": "#df7963"
        }[label]

        draw_ellipse(mean, cov, ax, ellipse_color, scale_factor=2)

        ax.scatter(
            mean[0], mean[1],
            marker=(8, 2, 0),
            s=300,
            color=group_colors[label],
            edgecolor='black',
            zorder=5
        )

    # Lesional severity overlay with explicit labels
    severity_label_map = {
        'Low': 'Acne Lesional Low (1–2)',
        'Moderate': 'Acne Lesional Mod (3–4)',
        'High': 'Acne Lesional High (5–6)'
    }

    lesional = spca_df[spca_df['category_label'] == 'Acne Lesional']
    for sev, color in severity_colors.items():
        sev_data = lesional[lesional['lesional_score'] == sev]
        if sev_data.empty:
            continue

        sns.scatterplot(
            data=sev_data,
            x='PC1',
            y='PC2',
            facecolor=color,
            edgecolor='k',
            linewidth=0.5,
            s=100,
            ax=ax,
            label=severity_label_map[sev]
        )

    # Manually set legend location per plot
    if file_suffix == "V1-V3":
        ax.legend(
            bbox_to_anchor=(0.98, 0.165),
            frameon=True,
            fontsize=11
        )
    elif file_suffix == "V4":
        ax.legend(
            bbox_to_anchor=(0.46, 0.25),
            frameon=True,
            fontsize=11
        )
    else:
        ax.legend(loc="best", frameon=True, fontsize=12)

    # Axes
    ax.set_xlabel(
        f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)',
        fontsize=18
    )
    ax.set_ylabel(
        f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)',
        fontsize=18
    )

    ax.tick_params(labelsize=14)

    # Title
    title_map = {
        'RPCA - 16S rRNA (V1-V3)': '16S rRNA (V1-V3) rPCA Beta Diversity',
        'RPCA - 16S rRNA (V4)': '16S rRNA (V4) rPCA Beta Diversity'
    }
    ax.set_title(title_map.get(plot_title, plot_title), fontsize=20)

    # PERMANOVA text with significance stars
    short = {
        "Healthy": "H",
        "Acne Non-lesional": "ANL",
        "Acne Lesional": "AL"
    }

    def pval_prefix(p):
        if p < 0.001:
            return "*** "
        elif p < 0.01:
            return "** "
        elif p < 0.05:
            return "* "
        else:
            return ""

    ptext = "\n".join(
        f"{short[r['Group1']]} vs {short[r['Group2']]}: "
        f"{pval_prefix(r['p-value'])}p={r['p-value']:.3f}; "
        f"F={r['Test Statistic']:.3f}"
        for _, r in results_df.iterrows()
    )

    ax.text(
        ax.get_xlim()[0] + 0.01,
        ax.get_ylim()[0] + 0.01,
        ptext,
        fontsize=12,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

    plt.tight_layout()
    plt.savefig(
        os.path.join(output_path, f"rPCA_beta_diversity_{file_suffix}.png"),
        dpi=600
    )
    plt.close(fig)


In [12]:
for table, metadata, plot_title, file_suffix, output_path in tables_metadata:
    ordination, distance_matrix, spca_df, results_df = run_rpca_analysis(
        table,
        metadata
    )

    plot_rpca_results(
        ordination,
        spca_df,
        results_df,
        plot_title,
        output_path,
        file_suffix,
        group_order,
        group_colors,
        severity_colors
    )

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_85344/1895773851.py:111: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  f'PC1 ({ordination.proportion_explained[0] * 100:.2f}%)',
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_85344/1895773851.py:115: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  f'PC2 ({ordination.proportion_explained[1] * 100:.2f}%)',
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_85344/1895773851.py:111: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with Data